In [1]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


#### Busca Dados INPC via código Atualiza-Indicador

In [3]:

from database.conexao import conectar
from indicadores.cdi import Cdi

with conectar():
    dados_cdi = Cdi.buscar()
    for aa in dados_cdi:
        print(aa)


{'data': '01/10/2026', 'valor': '0.56'}
{'data': '01/09/2026', 'valor': '1.08'}
{'data': '01/08/2026', 'valor': '1.09'}
{'data': '01/07/2026', 'valor': '1.22'}
{'data': '01/06/2026', 'valor': '1.12'}
{'data': '01/05/2026', 'valor': '1.07'}


#### Busca Dados INPC via código Atualiza-Indicador

In [4]:

from database.conexao import conectar
from indicadores.cdi import Cdi

with conectar():
    dados_cdi = Cdi.select()
    for aa in dados_cdi:
        print(aa.indice)
        print(aa.dt_referencia)


#### Busca Dados INPC Diretamente da API

In [5]:

ultimos_meses = 6

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados/ultimos/{ultimos_meses}?formato=json"

#parametros = { "formato": "json", "dataInicial": data_inicial, "dataFinal": data_final }

resposta = requests.get(url, timeout=10)
resposta.raise_for_status()
dados = resposta.json() # lista de dicionários

# Percorrer a lista de dicionário
for cdi in dados:
    print(cdi)


{'data': '01/10/2026', 'valor': '0.56'}
{'data': '01/09/2026', 'valor': '1.08'}
{'data': '01/08/2026', 'valor': '1.09'}
{'data': '01/07/2026', 'valor': '1.22'}
{'data': '01/06/2026', 'valor': '1.12'}
{'data': '01/05/2026', 'valor': '1.07'}


#### Gera datas

In [6]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=230)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

print(f"Período: {data_inicial} até {data_final}")


Período: 03/03/2026 até 19/10/2026


#### Busca Dados INPC da API com parêmetro de datas

In [7]:

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários

# Percorrer a lista de dicionário
for cdi in dados:
    print(cdi)


{'data': '01/03/2026', 'valor': '1.21'}
{'data': '01/04/2026', 'valor': '1.09'}
{'data': '01/05/2026', 'valor': '1.07'}
{'data': '01/06/2026', 'valor': '1.12'}
{'data': '01/07/2026', 'valor': '1.22'}
{'data': '01/08/2026', 'valor': '1.09'}
{'data': '01/09/2026', 'valor': '1.08'}
{'data': '01/10/2026', 'valor': '0.56'}


#### Insere Dados no Banco (transformando dados)

In [8]:
# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários

# Transforma dados para o formato da tabela Igpm
dados_cdi = []
for registro in dados:
    dados_cdi.append({"indice": registro["valor"],
                       "status": True,
                       "dt_referencia": datetime.strptime(registro["data"], "%d/%m/%Y").date()
                      })

with conectar():
    Cdi.insert_many(dados_cdi).on_conflict_ignore().execute()

#### Visualiza DADOS INPC (lista de dicionário)

In [9]:
for cdi in dados:
    print(cdi["data"], cdi["valor"])


01/03/2026 1.21
01/04/2026 1.09
01/05/2026 1.07
01/06/2026 1.12
01/07/2026 1.22
01/08/2026 1.09
01/09/2026 1.08
01/10/2026 0.56


#### Carga direta no Banco (registro por registro)

In [10]:

from database.conexao import conectar
from indicadores.cdi import Cdi
from decimal import Decimal

with conectar():
    for registro in dados:
        Cdi.insert(indice=registro["valor"], status=True, dt_referencia=registro["data"]).on_conflict_ignore().execute()


#### Carga direta no Banco (Em Massa)

In [11]:

from database.conexao import conectar
from indicadores.cdi import Cdi
from decimal import Decimal

with conectar():
    Cdi.insert_many(dados_cdi).on_conflict_ignore().execute()

#### Atualiza tabela INPC usando método do código Atualiza-Indicador

In [13]:

from database.conexao import conectar
from indicadores.cdi import Cdi

aa = Cdi.atualizar_cdi()

print(aa)

None
